# Этап 1. Data Audit и спецификация правил

Этот ноутбук закрывает первый этап проекта по кейсу ЦИТиС:

1. Загружаем и первично проверяем датасет.
2. Формируем профиль качества данных (пропуски, форматы, логические противоречия).
3. Формализуем правила контроля качества.
4. Строим таблицу `quality_flags` для дальнейших этапов (анализ нарушений, ML и дашборд).


## 1) Импорт библиотек

Используем `pandas`/`numpy` для анализа табличных данных и `re` для проверок формата полей.


In [ ]:
import re
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 160)


## 2) Загрузка данных

Читаем исходный CSV с разделителем `;`, сохраняем копию сырых данных (`df_raw`) и рабочую таблицу (`df`).


In [ ]:
DATA_PATH = '../hakaton.csv'

df_raw = pd.read_csv(DATA_PATH, sep=';', dtype=str)
df = df_raw.copy()

print('Rows:', len(df))
print('Columns:', len(df.columns))
df.head(3)


## 3) Базовая подготовка

- Приводим строки к единому виду (trim пробелов).
- Преобразуем даты в datetime.
- Нормализуем значения результата (`result_norm`).
- Формируем рабочий идентификатор ребенка `child_key` на основе ФИО + даты рождения.


In [ ]:
# trim по всем строковым колонкам
for col in df.columns:
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].replace({'': np.nan, 'nan': np.nan})

# даты
df['bdate_dt'] = pd.to_datetime(df['bdate'], errors='coerce')
df['test_date_dt'] = pd.to_datetime(df['test_date'], errors='coerce')

# результат
df['result_norm'] = df['result'].str.lower().str.strip()

# ключ ребенка
name_parts = ['last_name', 'first_name', 'middle_name', 'bdate']
df['child_key'] = (
    df[name_parts]
    .fillna('')
    .agg('|'.join, axis=1)
    .str.replace(r'\s+', ' ', regex=True)
)

print('Уникальных child_key:', df['child_key'].nunique())
print('Распределение result_norm:')
print(df['result_norm'].value_counts(dropna=False))


## 4) Профиль качества данных

Считаем долю пропусков по полям и выделяем потенциально проблемные зоны.


In [ ]:
missing_profile = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_share': (df.isna().mean() * 100).round(2)
}).sort_values('missing_count', ascending=False)

missing_profile.head(20)


## 5) Спецификация правил качества (Rule Book)

Ниже фиксируется набор правил, который будет использоваться в pipeline.

**Критичные правила** (влияют на корректность анализа):
- `FREQ_LT_90_DAYS` — повторный тест ранее 90 дней.
- `TEST_BEFORE_BIRTH` — дата теста раньше даты рождения.
- `MISSING_CORE_FIELDS` — пропуски в критичных полях (ФИО/даты/результат).
- `INVALID_RESULT_VALUE` — результат не в {достаточный, недостаточный}.

**Высокий приоритет**:
- `INVALID_CLASS_RANGE` — класс вне диапазона 1..11.
- `AGE_CLASS_MISMATCH` — несоответствие возраста и класса.
- `ID_DOC_INCONSISTENT_PER_CHILD` — у одного ребенка несколько разных `id_doc`.

**Средний приоритет**:
- `NON_NUMERIC_VARIANT` — нестандартный формат `variant`.
- `POTENTIAL_DUPLICATE_RECORD` — потенциальный дубль записи.
- `MISSING_NONCRITICAL_FIELDS` — пропуски во второстепенных полях.


## 6) Расчет флагов качества

Вычисляем флаги по каждому правилу и сохраняем их как отдельные бинарные признаки (0/1).


In [ ]:
# 6.1 Критичные пропуски
core_fields = ['last_name', 'first_name', 'bdate', 'test_date', 'result']

df['flag_missing_core_fields'] = df[core_fields].isna().any(axis=1).astype(int)

# 6.2 Некорректное значение результата
valid_results = {'достаточный', 'недостаточный'}
df['flag_invalid_result_value'] = (~df['result_norm'].isin(valid_results)).astype(int)

# 6.3 test_date раньше bdate
df['flag_test_before_birth'] = (
    (df['test_date_dt'].notna()) &
    (df['bdate_dt'].notna()) &
    (df['test_date_dt'] < df['bdate_dt'])
).astype(int)

# 6.4 Класс вне диапазона 1..11
class_num = pd.to_numeric(df['class'], errors='coerce')
df['flag_invalid_class_range'] = (
    class_num.isna() | ~class_num.between(1, 11)
).astype(int)

# 6.5 Несоответствие возраста и класса (эвристика: класс+4 <= возраст <= класс+10)
age_years = ((df['test_date_dt'] - df['bdate_dt']).dt.days / 365.25)
df['flag_age_class_mismatch'] = (
    class_num.notna() &
    age_years.notna() &
    ((age_years < (class_num + 4)) | (age_years > (class_num + 10)))
).astype(int)

# 6.6 Ненормативный variant (ожидаем цифры)
df['flag_non_numeric_variant'] = (~df['variant'].fillna('').str.fullmatch(r'\d+')).astype(int)

# 6.7 Несколько id_doc у одного child_key
id_doc_per_child = (
    df.assign(id_doc_norm=df['id_doc'].fillna('').str.strip())
      .query("id_doc_norm != ''")
      .groupby('child_key')['id_doc_norm']
      .nunique()
)
children_multi_id = set(id_doc_per_child[id_doc_per_child > 1].index)
df['flag_id_doc_inconsistent_per_child'] = df['child_key'].isin(children_multi_id).astype(int)

# 6.8 Потенциальные дубли (без учета our_number)
dupe_key = [
    'last_name', 'first_name', 'middle_name', 'bdate', 'id_doc', 'test_date',
    'variant', 'class', 'result_norm', 'ogrn_naprav', 'ogrn_area'
]
df['flag_potential_duplicate_record'] = df.duplicated(subset=dupe_key, keep=False).astype(int)


## 7) Проверка частоты тестирования (ключевое правило кейса)

Для каждого ребенка сортируем тесты по дате, считаем интервал до предыдущего теста и флагуем интервалы `< 90` дней.


In [ ]:
df = df.sort_values(['child_key', 'test_date_dt', 'our_number']).copy()
df['prev_test_date_dt'] = df.groupby('child_key')['test_date_dt'].shift(1)
df['days_since_prev_test'] = (df['test_date_dt'] - df['prev_test_date_dt']).dt.days

df['flag_freq_lt_90_days'] = (
    df['days_since_prev_test'].notna() &
    (df['days_since_prev_test'] < 90)
).astype(int)

# child-level признак: у ребенка есть хотя бы одно нарушение
child_freq_violation = df.groupby('child_key')['flag_freq_lt_90_days'].max()
viol_children = int((child_freq_violation == 1).sum())

print('Нарушающих интервалов (<90 дней):', int(df['flag_freq_lt_90_days'].sum()))
print('Детей с >=1 нарушением:', viol_children)


## 8) Сводка флагов

На этом шаге получаем финальный профиль качества и определяем приоритеты исправления для организаторов.


In [ ]:
flag_cols = [c for c in df.columns if c.startswith('flag_')]

flags_summary = (
    pd.DataFrame({
        'flag': flag_cols,
        'count': [int(df[c].sum()) for c in flag_cols]
    })
    .assign(share_pct=lambda x: (x['count'] / len(df) * 100).round(2))
    .sort_values('count', ascending=False)
)

flags_summary


## 9) Таблица quality_flags для следующих этапов

Сохраняем «узкую» таблицу флагов, пригодную для:
- построения дашборда качества,
- приоритизации ручной проверки,
- обучения baseline-модели риска.


In [ ]:
quality_flags = df[[
    'our_number', 'child_key', 'test_date', 'result_norm', 'days_since_prev_test',
    *flag_cols
]].copy()

quality_flags['quality_score'] = quality_flags[flag_cols].sum(axis=1)

# Категоризация записи по уровню риска
quality_flags['risk_level'] = pd.cut(
    quality_flags['quality_score'],
    bins=[-1, 0, 2, 99],
    labels=['low', 'medium', 'high']
)

quality_flags.head(10)


## 10) Экспорт артефактов этапа 1

Экспортируем 2 таблицы:
1. `quality_flags_stage1.csv` — флаги на уровне записей.
2. `flags_summary_stage1.csv` — агрегированная статистика по типам ошибок.


In [ ]:
OUT_FLAGS = '../quality_flags_stage1.csv'
OUT_SUMMARY = '../flags_summary_stage1.csv'

quality_flags.to_csv(OUT_FLAGS, index=False)
flags_summary.to_csv(OUT_SUMMARY, index=False)

print('Saved:', OUT_FLAGS)
print('Saved:', OUT_SUMMARY)


## 11) Выводы этапа 1

После выполнения ноутбука у нас есть:
- формализованный набор правил контроля качества;
- воспроизводимый pipeline флагов аномалий;
- готовая витрина для Этапа 2 (нормализация/дедупликация) и Этапа 3 (реестр нарушений частоты).

Следующим шагом можно перейти к Этапу 2 и сделать:
1. более устойчивый `child_key` (с учетом ошибок ввода ФИО/документов),
2. стратегию merge/resolve для дублей,
3. нормализованные справочники школ и вариантов теста.


# Этап 2. Нормализация и дедупликация

На этом этапе строим более устойчивую структуру данных:

1. Нормализуем идентификаторы ребенка (`child_key_stage2`).
2. Ищем и маркируем потенциальные кластеры дублей.
3. Формируем каноническую запись на кластер.
4. Готовим витрину `records_stage2` для следующих этапов.

## 12) Нормализация идентификаторов

Здесь создаем нормализованные поля (`*_norm`) и строим `child_key_stage2` по приоритету:

- сначала документ ребенка (`id_doc_norm`) если он есть,
- иначе fallback на ФИО + дата рождения.

Так мы снижаем влияние вариативности записи ФИО.

In [ ]:
def norm_text(x: str) -> str:
    if pd.isna(x):
        return ''
    x = str(x).strip().upper()
    x = re.sub(r'\s+', ' ', x)
    x = x.replace('Ё', 'Е')
    return x

# нормализация базовых атрибутов
df['last_name_norm'] = df['last_name'].apply(norm_text)
df['first_name_norm'] = df['first_name'].apply(norm_text)
df['middle_name_norm'] = df['middle_name'].fillna('').apply(norm_text)
df['id_doc_norm'] = df['id_doc'].fillna('').apply(norm_text)
df['guard_id_doc_norm'] = df['guard_id_doc'].fillna('').apply(norm_text)

# очищаем id_doc от служебных символов, оставляем буквы/цифры
for col in ['id_doc_norm', 'guard_id_doc_norm']:
    df[col] = df[col].str.replace(r'[^0-9A-ZА-Я]', '', regex=True)

# fallback key на случай отсутствия id_doc
fallback_name_key = (
    df['last_name_norm'] + '|' +
    df['first_name_norm'] + '|' +
    df['middle_name_norm'] + '|' +
    df['bdate'].fillna('')
)

df['child_key_stage2'] = np.where(
    df['id_doc_norm'].str.len() > 0,
    'DOC|' + df['id_doc_norm'],
    'NAME|' + fallback_name_key
)

print('child_key (stage1):', df['child_key'].nunique())
print('child_key_stage2:', df['child_key_stage2'].nunique())

## 13) Поиск дублей и конфликтов

Делаем два типа группировок:

1. `duplicate_group_id` — потенциальные точные дубли по ключевым полям события.
2. `conflict_group_id` — конфликтные записи внутри одного `child_key_stage2` и одной даты теста (например, разные result/class/variant).

Это разделяет «чистые дубли» и «противоречивые версии» одной записи.

In [ ]:
# 13.1 Потенциальные точные дубли
exact_dupe_key_stage2 = [
    'child_key_stage2', 'test_date', 'variant', 'class', 'result_norm',
    'ogrn_naprav', 'ogrn_area'
]

dupe_group_size = df.groupby(exact_dupe_key_stage2)['our_number'].transform('size')
df['is_exact_duplicate_stage2'] = (dupe_group_size > 1).astype(int)

df['duplicate_group_id'] = np.where(
    dupe_group_size > 1,
    df[exact_dupe_key_stage2].astype(str).agg('|'.join, axis=1),
    np.nan
)

# 13.2 Конфликтные версии: один ребенок + одна дата, но разные result/class/variant
conflict_scope = ['child_key_stage2', 'test_date']
conflict_signature = ['result_norm', 'class', 'variant']

unique_signature_per_scope = (
    df.groupby(conflict_scope)[conflict_signature]
      .apply(lambda x: x.astype(str).drop_duplicates().shape[0])
      .rename('sig_cnt')
      .reset_index()
)

conflict_scopes = unique_signature_per_scope.query('sig_cnt > 1')[conflict_scope]
conflict_scopes['conflict_group_id'] = conflict_scopes.astype(str).agg('|'.join, axis=1)

df = df.merge(conflict_scopes, on=conflict_scope, how='left')
df['is_conflict_version_stage2'] = df['conflict_group_id'].notna().astype(int)

print('Записей в точных дублях:', int(df['is_exact_duplicate_stage2'].sum()))
print('Записей в конфликтных версиях:', int(df['is_conflict_version_stage2'].sum()))

## 14) Стратегия выбора канонической записи (dedup resolve)

Внутри каждого кластера дубликатов/конфликтов выбираем одну каноническую запись:

- приоритет 1: запись с меньшим `quality_score`,
- приоритет 2: запись без конфликтного статуса,
- приоритет 3: более поздняя дата теста,
- приоритет 4: лексикографически минимальный `our_number` как стабильный tie-breaker.

Остальные строки сохраняем в журнал `dedup_decisions` с причиной исключения.

In [ ]:
# Формируем единый ключ группы для dedup resolve
# Если есть duplicate_group_id, используем его.
# Иначе если есть conflict_group_id, используем его.
# Иначе группа = сама запись (our_number).
df['dedup_group_id'] = np.where(
    df['duplicate_group_id'].notna(),
    'DUP|' + df['duplicate_group_id'].astype(str),
    np.where(
        df['conflict_group_id'].notna(),
        'CONFLICT|' + df['conflict_group_id'].astype(str),
        'SINGLE|' + df['our_number'].astype(str)
    )
)

# Правила ранжирования внутри группы
rank_df = df.copy()
rank_df['qscore'] = rank_df['quality_score'].fillna(0)
rank_df['conflict_penalty'] = rank_df['is_conflict_version_stage2']
rank_df['test_date_rank'] = rank_df['test_date_dt']

rank_df = rank_df.sort_values(
    by=['dedup_group_id', 'qscore', 'conflict_penalty', 'test_date_rank', 'our_number'],
    ascending=[True, True, True, False, True]
)

# Первая запись в группе = каноническая
rank_df['dedup_rank'] = rank_df.groupby('dedup_group_id').cumcount() + 1
rank_df['is_canonical_stage2'] = (rank_df['dedup_rank'] == 1).astype(int)

records_stage2 = rank_df[rank_df['is_canonical_stage2'] == 1].copy()

dedup_decisions = rank_df[[
    'our_number', 'dedup_group_id', 'dedup_rank', 'is_canonical_stage2',
    'is_exact_duplicate_stage2', 'is_conflict_version_stage2', 'quality_score'
]].copy()

dedup_decisions['drop_reason'] = np.where(
    dedup_decisions['is_canonical_stage2'] == 1,
    'kept_canonical',
    np.where(
        dedup_decisions['is_exact_duplicate_stage2'] == 1,
        'dropped_exact_duplicate',
        np.where(
            dedup_decisions['is_conflict_version_stage2'] == 1,
            'dropped_conflict_version',
            'dropped_by_rank'
        )
    )
)

print('До dedup:', len(df))
print('После dedup (records_stage2):', len(records_stage2))
print('Удалено:', len(df) - len(records_stage2))

## 15) Контрольные метрики этапа 2

Считаем итоговые показатели, чтобы понять, насколько стабилизировались данные после нормализации и дедупликации.

In [ ]:
stage2_metrics = pd.DataFrame([
    {'metric': 'rows_before', 'value': len(df)},
    {'metric': 'rows_after', 'value': len(records_stage2)},
    {'metric': 'rows_removed', 'value': len(df) - len(records_stage2)},
    {'metric': 'exact_duplicate_rows', 'value': int(df['is_exact_duplicate_stage2'].sum())},
    {'metric': 'conflict_version_rows', 'value': int(df['is_conflict_version_stage2'].sum())},
    {'metric': 'unique_child_stage1', 'value': df['child_key'].nunique()},
    {'metric': 'unique_child_stage2', 'value': df['child_key_stage2'].nunique()},
])

stage2_metrics

## 16) Экспорт артефактов этапа 2

Сохраняем три таблицы:

1. `records_stage2.csv` — очищенные канонические записи.
2. `dedup_decisions_stage2.csv` — журнал решений dedup.
3. `stage2_metrics.csv` — контрольные метрики этапа.

In [ ]:
OUT_RECORDS_STAGE2 = '../records_stage2.csv'
OUT_DEDUP_DECISIONS = '../dedup_decisions_stage2.csv'
OUT_STAGE2_METRICS = '../stage2_metrics.csv'

records_stage2.to_csv(OUT_RECORDS_STAGE2, index=False)
dedup_decisions.to_csv(OUT_DEDUP_DECISIONS, index=False)
stage2_metrics.to_csv(OUT_STAGE2_METRICS, index=False)

print('Saved:', OUT_RECORDS_STAGE2)
print('Saved:', OUT_DEDUP_DECISIONS)
print('Saved:', OUT_STAGE2_METRICS)

# Этап 3. Реестр нарушений частоты тестирования

На этом этапе используем очищенную витрину `records_stage2` и строим:

1. Реестр нарушений правила `< 90 дней` на уровне событий.
2. Агрегации по ребенку, школам и временным периодам.
3. Приоритетный список для контрольных мероприятий.

## 17) Подготовка витрины для проверки частоты

Сортируем тесты внутри `child_key_stage2`, считаем интервал до предыдущего тестирования и флаг нарушения (`< 90` дней).

Также выделяем градации срочности:
- `critical` — интервал < 30 дней,
- `high` — 30..59 дней,
- `medium` — 60..89 дней.

In [ ]:
# Берем очищенную витрину этапа 2
freq_df = records_stage2.copy()

freq_df = freq_df.sort_values(['child_key_stage2', 'test_date_dt', 'our_number']).copy()
freq_df['prev_test_date_dt_stage3'] = freq_df.groupby('child_key_stage2')['test_date_dt'].shift(1)
freq_df['days_since_prev_test_stage3'] = (freq_df['test_date_dt'] - freq_df['prev_test_date_dt_stage3']).dt.days

freq_df['flag_freq_violation_stage3'] = (
    freq_df['days_since_prev_test_stage3'].notna() &
    (freq_df['days_since_prev_test_stage3'] < 90)
).astype(int)

freq_df['violation_severity'] = pd.cut(
    freq_df['days_since_prev_test_stage3'],
    bins=[-np.inf, 29, 59, 89, np.inf],
    labels=['critical', 'high', 'medium', 'ok']
)

# Для строк без предыдущего теста severity не применима
freq_df.loc[freq_df['days_since_prev_test_stage3'].isna(), 'violation_severity'] = np.nan

print('Всего записей (после stage2):', len(freq_df))
print('Нарушающих интервалов (<90):', int(freq_df['flag_freq_violation_stage3'].sum()))
print(freq_df['violation_severity'].value_counts(dropna=False))

## 18) Реестр нарушений (event-level)

В реестр попадают только события-нарушения (`flag_freq_violation_stage3 = 1`).

Для каждого события сохраняем контекст:
- текущий и предыдущий тест,
- интервал между тестами,
- признаки качества,
- источники направившей и площадки тестирования.

In [ ]:
violations_registry = freq_df.loc[freq_df['flag_freq_violation_stage3'] == 1, [
    'our_number',
    'child_key_stage2',
    'test_date',
    'prev_test_date_dt_stage3',
    'days_since_prev_test_stage3',
    'violation_severity',
    'result_norm',
    'class',
    'variant',
    'ogrn_naprav',
    'name_naprav',
    'ogrn_area',
    'name_area',
    'quality_score',
    'risk_level'
]].copy()

violations_registry = violations_registry.rename(columns={
    'test_date': 'current_test_date',
    'prev_test_date_dt_stage3': 'prev_test_date'
})

violations_registry = violations_registry.sort_values(
    ['violation_severity', 'days_since_prev_test_stage3', 'quality_score', 'current_test_date'],
    ascending=[True, True, False, True]
)

violations_registry.head(15)

## 19) Child-level сводка нарушений

Агрегируем нарушения на уровне ребенка:
- число нарушающих событий,
- минимальный интервал,
- последняя дата нарушения,
- максимальная тяжесть.

Это нужно для приоритизации кейсов при ручной проверке.

In [ ]:
severity_rank = {'critical': 3, 'high': 2, 'medium': 1}

child_violations = violations_registry.copy()
child_violations['severity_rank'] = child_violations['violation_severity'].map(severity_rank).fillna(0)

violations_by_child = (
    child_violations
    .groupby('child_key_stage2', as_index=False)
    .agg(
        violations_count=('our_number', 'count'),
        min_interval_days=('days_since_prev_test_stage3', 'min'),
        last_violation_date=('current_test_date', 'max'),
        max_severity_rank=('severity_rank', 'max')
    )
)

severity_rank_reverse = {3: 'critical', 2: 'high', 1: 'medium', 0: 'unknown'}
violations_by_child['max_severity'] = violations_by_child['max_severity_rank'].map(severity_rank_reverse)

violations_by_child = violations_by_child.sort_values(
    ['max_severity_rank', 'violations_count', 'min_interval_days'],
    ascending=[False, False, True]
)

violations_by_child.head(15)

## 20) School-level мониторинг

Формируем рейтинг организаций по доле нарушений:

- для направивших школ (`name_naprav`),
- для площадок тестирования (`name_area`).

Добавляем фильтр минимального числа тестов, чтобы убрать шум малых выборок.

In [ ]:
MIN_TESTS_THRESHOLD = 20

# Направившие школы
school_naprav_stats = (
    freq_df.groupby(['ogrn_naprav', 'name_naprav'], as_index=False)
    .agg(
        tests_total=('our_number', 'count'),
        violations_total=('flag_freq_violation_stage3', 'sum')
    )
)
school_naprav_stats['violation_share_pct'] = (school_naprav_stats['violations_total'] / school_naprav_stats['tests_total'] * 100).round(2)
school_naprav_stats = school_naprav_stats.query('tests_total >= @MIN_TESTS_THRESHOLD').sort_values('violation_share_pct', ascending=False)

# Площадки тестирования
school_area_stats = (
    freq_df.groupby(['ogrn_area', 'name_area'], as_index=False)
    .agg(
        tests_total=('our_number', 'count'),
        violations_total=('flag_freq_violation_stage3', 'sum')
    )
)
school_area_stats['violation_share_pct'] = (school_area_stats['violations_total'] / school_area_stats['tests_total'] * 100).round(2)
school_area_stats = school_area_stats.query('tests_total >= @MIN_TESTS_THRESHOLD').sort_values('violation_share_pct', ascending=False)

print('Top-10 направивших школ по доле нарушений:')
display(school_naprav_stats.head(10))
print('Top-10 площадок тестирования по доле нарушений:')
display(school_area_stats.head(10))

## 21) Временная динамика нарушений

Строим monthly-агрегацию, чтобы понимать сезонность и пики нарушений. Это напрямую пригодится для графиков на дашборде.

In [ ]:
freq_df['test_month'] = freq_df['test_date_dt'].dt.to_period('M').astype(str)

monthly_violation_stats = (
    freq_df.groupby('test_month', as_index=False)
    .agg(
        tests_total=('our_number', 'count'),
        violations_total=('flag_freq_violation_stage3', 'sum')
    )
    .sort_values('test_month')
)
monthly_violation_stats['violation_share_pct'] = (
    monthly_violation_stats['violations_total'] / monthly_violation_stats['tests_total'] * 100
).round(2)

monthly_violation_stats

## 22) Экспорт артефактов этапа 3

Сохраняем отдельные витрины под дашборд и контроль:

1. `violations_registry_stage3.csv` — реестр событий-нарушений.
2. `violations_by_child_stage3.csv` — агрегат по ребенку.
3. `school_naprav_stats_stage3.csv` — статистика направивших школ.
4. `school_area_stats_stage3.csv` — статистика площадок тестирования.
5. `monthly_violation_stats_stage3.csv` — динамика по месяцам.
6. `stage3_metrics.csv` — ключевые KPI этапа.

In [ ]:
stage3_metrics = pd.DataFrame([
    {'metric': 'records_stage2_total', 'value': len(freq_df)},
    {'metric': 'violations_events_total', 'value': int(freq_df['flag_freq_violation_stage3'].sum())},
    {'metric': 'violations_children_total', 'value': violations_by_child['child_key_stage2'].nunique()},
    {'metric': 'violation_share_pct', 'value': round(freq_df['flag_freq_violation_stage3'].mean() * 100, 2)},
    {'metric': 'critical_events_total', 'value': int((violations_registry['violation_severity'] == 'critical').sum())},
    {'metric': 'high_events_total', 'value': int((violations_registry['violation_severity'] == 'high').sum())},
    {'metric': 'medium_events_total', 'value': int((violations_registry['violation_severity'] == 'medium').sum())}
])

OUT_VIOL_REG = '../violations_registry_stage3.csv'
OUT_VIOL_CHILD = '../violations_by_child_stage3.csv'
OUT_SCHOOL_NAPRAV = '../school_naprav_stats_stage3.csv'
OUT_SCHOOL_AREA = '../school_area_stats_stage3.csv'
OUT_MONTHLY = '../monthly_violation_stats_stage3.csv'
OUT_STAGE3_METRICS = '../stage3_metrics.csv'

violations_registry.to_csv(OUT_VIOL_REG, index=False)
violations_by_child.to_csv(OUT_VIOL_CHILD, index=False)
school_naprav_stats.to_csv(OUT_SCHOOL_NAPRAV, index=False)
school_area_stats.to_csv(OUT_SCHOOL_AREA, index=False)
monthly_violation_stats.to_csv(OUT_MONTHLY, index=False)
stage3_metrics.to_csv(OUT_STAGE3_METRICS, index=False)

print('Saved:', OUT_VIOL_REG)
print('Saved:', OUT_VIOL_CHILD)
print('Saved:', OUT_SCHOOL_NAPRAV)
print('Saved:', OUT_SCHOOL_AREA)
print('Saved:', OUT_MONTHLY)
print('Saved:', OUT_STAGE3_METRICS)

stage3_metrics

# Этап 4. Baseline ML для риск-скоринга

Цель этапа — построить baseline-модель, которая оценивает риск записи/события как потенциально проблемного для контроля качества и соблюдения процедуры.

В качестве целевой переменной используем `flag_freq_violation_stage3` (нарушение интервала < 90 дней).

## 23) Подготовка ML-выборки

Используем `freq_df` из Этапа 3 и формируем признаки:

- числовые,
- категориальные,
- исторические (частота/качество).

Важно: исключаем явную утечку таргета (сам `days_since_prev_test_stage3` в модель не подаем).

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report

ml_df = freq_df.copy()

# Целевая переменная
y = ml_df['flag_freq_violation_stage3'].astype(int)

# Простые инженерные признаки без утечки
ml_df['test_month_num'] = ml_df['test_date_dt'].dt.month
ml_df['test_weekday'] = ml_df['test_date_dt'].dt.weekday
ml_df['child_attempt_no'] = ml_df.groupby('child_key_stage2').cumcount() + 1
ml_df['child_total_attempts'] = ml_df.groupby('child_key_stage2')['our_number'].transform('count')
ml_df['prev_quality_score'] = ml_df.groupby('child_key_stage2')['quality_score'].shift(1)
ml_df['prev_result_norm'] = ml_df.groupby('child_key_stage2')['result_norm'].shift(1)
ml_df['class_num_feature'] = pd.to_numeric(ml_df['class'], errors='coerce')

numeric_features = [
    'quality_score', 'class_num_feature', 'child_attempt_no', 'child_total_attempts',
    'test_month_num', 'test_weekday', 'prev_quality_score'
]

categorical_features = [
    'result_norm', 'prev_result_norm', 'variant',
    'ogrn_naprav', 'ogrn_area', 'risk_level'
]

X = ml_df[numeric_features + categorical_features].copy()

print('ML dataset rows:', len(X))
print('Target share (positive):', round(y.mean() * 100, 2), '%')

## 24) Обучение baseline-модели

Пайплайн:

- иммутация пропусков,
- one-hot кодирование категориальных признаков,
- `HistGradientBoostingClassifier` как устойчивый baseline.

Метрики:
- ROC-AUC,
- PR-AUC (average precision),
- classification report при пороге 0.5.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

model = LogisticRegression(
    max_iter=2000,
    class_weight='balanced',
    random_state=42
)

clf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', model)
])

clf.fit(X_train, y_train)

y_proba = clf.predict_proba(X_test)[:, 1]
y_pred = (y_proba >= 0.5).astype(int)

roc_auc = roc_auc_score(y_test, y_proba)
pr_auc = average_precision_score(y_test, y_proba)

print('ROC-AUC:', round(roc_auc, 4))
print('PR-AUC :', round(pr_auc, 4))
print('\nClassification report (threshold=0.5):')
print(classification_report(y_test, y_pred, digits=4))

## 25) Интерпретация модели (важность факторов)

Для логистической регрессии используем абсолютное значение коэффициентов как baseline-интерпретацию вклада признаков.

In [ ]:
feature_names = clf.named_steps['preprocessor'].get_feature_names_out()
coefs = clf.named_steps['model'].coef_[0]

feature_importance = pd.DataFrame({
    'feature': feature_names,
    'coef': coefs,
    'abs_coef': np.abs(coefs)
}).sort_values('abs_coef', ascending=False)

feature_importance.head(20)

## 26) Скоринг всей витрины и экспорт артефактов этапа 4

Считаем `risk_score_ml` для всех записей `freq_df`, формируем ранжированный список high-risk кейсов и экспортируем результаты для дашборда и презентации.

In [ ]:
scoring_df = ml_df.copy()
scoring_df['risk_score_ml'] = clf.predict_proba(X)[:, 1]
scoring_df['risk_bucket_ml'] = pd.cut(
    scoring_df['risk_score_ml'],
    bins=[-0.01, 0.3, 0.7, 1.0],
    labels=['low', 'medium', 'high']
)

high_risk_cases = scoring_df.sort_values('risk_score_ml', ascending=False)[[
    'our_number', 'child_key_stage2', 'test_date', 'result_norm',
    'quality_score', 'risk_level', 'risk_score_ml', 'risk_bucket_ml',
    'ogrn_naprav', 'name_naprav', 'ogrn_area', 'name_area'
]].head(1000)

stage4_metrics = pd.DataFrame([
    {'metric': 'ml_rows_total', 'value': len(scoring_df)},
    {'metric': 'target_positive_share_pct', 'value': round(y.mean() * 100, 2)},
    {'metric': 'roc_auc', 'value': round(float(roc_auc), 4)},
    {'metric': 'pr_auc', 'value': round(float(pr_auc), 4)},
    {'metric': 'predicted_high_risk_rows', 'value': int((scoring_df['risk_bucket_ml'] == 'high').sum())},
    {'metric': 'predicted_medium_risk_rows', 'value': int((scoring_df['risk_bucket_ml'] == 'medium').sum())},
    {'metric': 'predicted_low_risk_rows', 'value': int((scoring_df['risk_bucket_ml'] == 'low').sum())}
])

OUT_SCORING = '../ml_scoring_stage4.csv'
OUT_HIGH_RISK = '../ml_high_risk_cases_stage4.csv'
OUT_IMPORTANCE = '../ml_feature_importance_stage4.csv'
OUT_STAGE4_METRICS = '../stage4_metrics.csv'

scoring_df.to_csv(OUT_SCORING, index=False)
high_risk_cases.to_csv(OUT_HIGH_RISK, index=False)
feature_importance.to_csv(OUT_IMPORTANCE, index=False)
stage4_metrics.to_csv(OUT_STAGE4_METRICS, index=False)

print('Saved:', OUT_SCORING)
print('Saved:', OUT_HIGH_RISK)
print('Saved:', OUT_IMPORTANCE)
print('Saved:', OUT_STAGE4_METRICS)

stage4_metrics

# Этап 4 (обновление). Isolation Forest

По вашему запросу baseline обновлен на алгоритм **Isolation Forest** (по подходу из статьи):
- обучаемся без целевой метки,
- выделяем аномалии как `predict == -1`,
- используем `decision_function` для ранжирования риска.

In [ ]:
from sklearn.ensemble import IsolationForest

iso_df = freq_df.copy()
iso_df['test_month_num'] = iso_df['test_date_dt'].dt.month
iso_df['test_weekday'] = iso_df['test_date_dt'].dt.weekday
iso_df['child_attempt_no'] = iso_df.groupby('child_key_stage2').cumcount() + 1
iso_df['child_total_attempts'] = iso_df.groupby('child_key_stage2')['our_number'].transform('count')
iso_df['prev_quality_score'] = iso_df.groupby('child_key_stage2')['quality_score'].shift(1)
iso_df['prev_result_norm'] = iso_df.groupby('child_key_stage2')['result_norm'].shift(1)
iso_df['class_num_feature'] = pd.to_numeric(iso_df['class'], errors='coerce')

iso_numeric_features = [
    'quality_score', 'class_num_feature', 'child_attempt_no', 'child_total_attempts',
    'test_month_num', 'test_weekday', 'prev_quality_score'
]
iso_categorical_features = [
    'result_norm', 'prev_result_norm', 'variant', 'ogrn_naprav', 'ogrn_area', 'risk_level'
]

iso_X = iso_df[iso_numeric_features + iso_categorical_features].copy()

iso_preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[('imputer', SimpleImputer(strategy='median'))]), iso_numeric_features),
        ('cat', Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), iso_categorical_features)
    ]
)

iso_Xp = iso_preprocessor.fit_transform(iso_X)

iforest = IsolationForest(
    n_estimators=300,
    max_samples=4096,
    contamination=0.03,
    n_jobs=-1,
    random_state=42
)
iforest.fit(iso_Xp)

iso_df['iforest_decision'] = iforest.decision_function(iso_Xp)
iso_df['iforest_is_anomaly'] = np.where(iforest.predict(iso_Xp) == -1, 1, 0)
iso_df['iforest_risk_score'] = -iso_df['iforest_decision']

q1 = iso_df['iforest_risk_score'].quantile(0.70)
q2 = iso_df['iforest_risk_score'].quantile(0.90)
iso_df['iforest_risk_bucket'] = np.where(
    iso_df['iforest_risk_score'] >= q2,
    'high',
    np.where(iso_df['iforest_risk_score'] >= q1, 'medium', 'low')
)

iso_df[['iforest_is_anomaly', 'iforest_risk_score', 'iforest_risk_bucket']].head()

In [ ]:
iso_manual_cnt = int(iso_df['flag_freq_violation_stage3'].sum())
iso_model_cnt = int(iso_df['iforest_is_anomaly'].sum())
iso_overlap = int(((iso_df['flag_freq_violation_stage3'] == 1) & (iso_df['iforest_is_anomaly'] == 1)).sum())

iso_precision_vs_manual = iso_overlap / iso_model_cnt if iso_model_cnt else 0
iso_recall_vs_manual = iso_overlap / iso_manual_cnt if iso_manual_cnt else 0

stage4_iforest_metrics = pd.DataFrame([
    {'metric': 'rows_total', 'value': len(iso_df)},
    {'metric': 'manual_anomalies_total', 'value': iso_manual_cnt},
    {'metric': 'iforest_anomalies_total', 'value': iso_model_cnt},
    {'metric': 'overlap_manual_iforest', 'value': iso_overlap},
    {'metric': 'precision_vs_manual', 'value': round(iso_precision_vs_manual, 4)},
    {'metric': 'recall_vs_manual', 'value': round(iso_recall_vs_manual, 4)},
    {'metric': 'iforest_contamination', 'value': 0.03},
])

iso_top_cases = iso_df.sort_values('iforest_risk_score', ascending=False).head(1000)[[
    'our_number', 'child_key_stage2', 'test_date', 'result_norm', 'quality_score', 'risk_level',
    'iforest_risk_score', 'iforest_is_anomaly', 'iforest_risk_bucket',
    'flag_freq_violation_stage3', 'days_since_prev_test_stage3', 'name_naprav', 'name_area'
]]

iso_df.to_csv('../ml_scoring_stage4_iforest.csv', index=False)
iso_top_cases.to_csv('../ml_high_risk_cases_stage4_iforest.csv', index=False)
stage4_iforest_metrics.to_csv('../stage4_iforest_metrics.csv', index=False)

stage4_iforest_metrics